# Technique 3: Case 1


Run all cells to obtain every configuration. Convergence and `%%timeit` use separate cells. To repeat timings, rerun each method’s setup and timing cells, leaving its convergence cell unchanged. All timing cells run serially; integrators are constructed beforehand.


In [1]:
import os
import sys
import copy
import importlib
import json
import platform
from pathlib import Path

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)


In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

def save_trial(i, result, accepted, nfev):
    records.append(dict(trial=int(i), accepted=bool(accepted), nfev=int(nfev),
        status=int(result.status) if result is not None else None,
        residual_norm=float(np.linalg.norm(result.fun)) if result is not None else None,
        x=result.x.tolist() if result is not None else None))


{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13_heyoka7121_paper/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.12.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np
import heyoka as hy

# =========================
# Setup (SI Units)
# =========================
AU2m = 149597870700 # JPL 
day2s = 86400.0     # JPL

tof = 3534.0*day2s # [s]

m0 = 4000.0 # [kg]
Tmax = 0.32 # [N]
Isp = 3000  # [s]

mu = 132712440018*1e9   # [m3∕s2]
g0 = 9.8065             # [m∕s2]

# Initial and terminal MEE (p, f, g, h, k, L); terminal L includes five revolutions.
mee0 = [149554268256.46957, -0.003766786954568226, 0.016286832739416963, -7.702048962437313e-06, 6.183156622292267e-07, 1.5955219194574601]


meef = [232433083985.70743, 0.1530290696088389, -0.5199481742007113, 0.01618310223871937, 0.11813952745106715, 33.763869726200596]


# =========================
# Nondimensionalize
# =========================
# Nondimension units
LUnit = AU2m
TUnit = np.sqrt(LUnit * LUnit * LUnit / mu)
MUnit = m0

VUnit = LUnit / TUnit # VUnit to m/s
AUnit = VUnit / TUnit # AUnit to m/s^2

# x0_nu, xf_nu, t0_nu, tf_nu, m0_nu, mu_nu, c1=Tmax_nu, c2=Isp_nu*g0_nu 
x0_nu = mee0
xf_nu = meef
t0_nu = 0.0
tf_nu = t0_nu+tof
m0_nu = m0
mu_nu = mu
Tmax_nu = Tmax
Isp_nu = Isp
g0_nu = g0

x0_nu[0] = x0_nu[0] / LUnit
xf_nu[0] = xf_nu[0] / LUnit

t0_nu = t0_nu / TUnit
tf_nu = tf_nu / TUnit
m0_nu = m0_nu / MUnit
mu_nu = mu_nu * TUnit * TUnit / LUnit / LUnit / LUnit
Tmax_nu = Tmax_nu / MUnit / AUnit
Isp_nu = Isp_nu / TUnit
g0_nu = g0_nu / AUnit

c1 = Tmax_nu
c2 = Isp_nu*g0_nu


In [4]:
# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000


minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
minpack_factor  = 0.1
minpack_diag    = [1.0]*3 + [1.0]*3 + [1.0]*1
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)


In [5]:
# =========================
# Define the shooting function
# =========================
def shooting_minT(lam, out = False, Time_out = None):
    # lam[0:7] : initial co-states [lx,ly,lz,lvx,lvy,lvz,lm]
    # #[GPT-no log modify] lam[7] : Tmax itself (nondimensional), used directly in shooting
    # function uses global variables ta(mu_nu, c1, c2), t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    
    ta.time = t0_nu
    # #[GPT-no log modify] Use Tmax directly instead of eta = log(Tmax).
    Tmax_shooting = float(lam[7])

    # #[GPT-no log modify] Numerical protection: require positive thrust and
    # keep the full-thrust mass positive over the propagation interval.
    dt_guard = float(tf_nu - t0_nu)
    Tmax_mass_limit = np.inf if dt_guard <= 0.0 else float(m0_nu * c2 / dt_guard)
    valid_Tmax = (
        np.isfinite(Tmax_shooting)
        and (Tmax_shooting > 0.0)
        and (Tmax_shooting < Tmax_mass_limit)
    )
    if not valid_Tmax:
        if Time_out is not None:
            return None
        target = float(Tmax_nu)
        if (not np.isfinite(target)) or target <= 0.0 or target >= Tmax_mass_limit:
            target = 0.5 * Tmax_mass_limit if np.isfinite(Tmax_mass_limit) else 1.0
        scale = max(abs(target), 1.0e-12)
        penalty = np.zeros(8, dtype=float)
        penalty[-1] = ((Tmax_shooting - target) / scale) if np.isfinite(Tmax_shooting) else -1.0
        return penalty
    ta.pars[:] = [mu_nu, Tmax_shooting, c2]

    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[0:7]
    ta.state[14] = 0.0 # q(t0) = 0

    if Time_out is None: # for root function
        ta.propagate_until(tf_nu)

        # Construct 8 equations: 6 terminal errors + 1 terminal co-state lm = 0 + q(tf) = 1
        retval = [
            ta.state[0] - xf_nu[0],
            ta.state[1] - xf_nu[1],
            ta.state[2] - xf_nu[2],
            ta.state[3] - xf_nu[3],
            ta.state[4] - xf_nu[4],
            ta.state[5] - xf_nu[5],
            ta.state[13],
            ta.state[14] - 1.0,
        ]

        if out:
            print("  Final Mass:", ta.state[6]*MUnit, "  Tmax:", Tmax_shooting*MUnit*AUnit)

    if Time_out is not None:
        ta.propagate_until(Time_out)

        state_t = ta.state[0:14]
        rho_t  = float(rho_func(state_t, pars=ta.pars)[0])
        rho_fuel_t  = float(rho_fuel_func(state_t, pars=ta.pars)[0])

        return {
            "t":  Time_out,
            "SG": rho_t,
            "SG_fuel": rho_fuel_t
        }

    return retval


In [6]:
# =========================
# Define the shooting function
# =========================
def shooting_maxm0(lam, out = False, Time_out = None):
    # lam[0:6] : initial co-states [lx,ly,lz,lvx,lvy,lvz]
    # #[GPT-no log modify] lam[6] : m0 itself (nondimensional), used directly in shooting
    # function uses global variables ta(mu_nu, c1, c2), t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    
    ta.time = t0_nu
    ta.pars[:] = [mu_nu, c1, c2]

    # #[GPT-no log modify] Use m0 directly instead of eta = log(m0).
    m0_shooting = float(lam[6])

    # #[GPT-no log modify] Numerical protection: require positive initial mass and
    # keep mass positive over the full-thrust propagation interval.
    dt_guard = float(tf_nu - t0_nu)
    m0_mass_floor = max(0.0, float((c1 / c2) * dt_guard)) if c2 > 0.0 else np.inf
    valid_m0 = np.isfinite(m0_shooting) and (m0_shooting > m0_mass_floor)
    if not valid_m0:
        if Time_out is not None:
            return None
        target = float(m0_nu)
        if (not np.isfinite(target)) or target <= m0_mass_floor:
            target = max(1.0, 1.1 * m0_mass_floor)
        scale = max(abs(target), 1.0e-12)
        penalty = np.zeros(7, dtype=float)
        penalty[-1] = ((m0_shooting - target) / scale) if np.isfinite(m0_shooting) else -1.0
        return penalty
    ta.state[:7] = x0_nu + [m0_shooting]
    ta.state[7:13] = lam[0:6]
    ta.state[13] = 1.0

    if Time_out is None: # for root function
        ta.propagate_until(tf_nu)

        # Construct 7 equations: 6 terminal mee errors + 1 terminal co-state lm = 0
        retval = [
            ta.state[0] - xf_nu[0],
            ta.state[1] - xf_nu[1],
            ta.state[2] - xf_nu[2],
            ta.state[3] - xf_nu[3],
            ta.state[4] - xf_nu[4],
            ta.state[5] - xf_nu[5],
            ta.state[13],
        ]

        if out:
            print("  Final Mass:", ta.state[6]*MUnit, "  Initial Mass:", m0_shooting*MUnit)

    if Time_out is not None:
        ta.propagate_until(Time_out)

        state_t = ta.state[0:14]
        rho_t  = float(rho_func(state_t, pars=ta.pars)[0])
        rho_fuel_t  = float(rho_fuel_func(state_t, pars=ta.pars)[0])

        return {
            "t":  Time_out,
            "SG": rho_t,
            "SG_fuel": rho_fuel_t
        }

    return retval


## MINT, original scaling


In [7]:
import twobody_MEE_hamilton_log as model
ta = model.get_integrator(formulation='MinT', compact=False)
functions = model.get_functions('MinT')
rho_func = functions['rho_func']
rho_fuel_func = functions['rho_fuel_func']
ta.pars[:] = [mu_nu, c1, c2]


In [8]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
np.random.seed(322)

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_Tmax = np.inf

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = np.empty(8)
    x0[:6] = (np.random.random(6) - 0.5) * 2.0 * 1.0
    x0[6] = np.random.random() * 1.0  # λ_m ∈ (0, 1) 

    # #[GPT-no log modify] Positive Tmax guess; last shooting variable is Tmax itself.
    # x0[7] = Tmax_nu * (0.5 + np.random.random())          # test1: [0.5*Tmax_nu, 1.5*Tmax_nu] #[GPT-no log modify]
    x0[7] = Tmax_nu * (0.5 + np.random.random()*(2.0-0.5))  # test2: [0.5*Tmax_nu, 2.0*Tmax_nu] #[GPT-no log modify]
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [9]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]

    time_start = time.perf_counter()

    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        # Call one more time with out = True
        shooting_minT(r['x'], out=True, Time_out=None)

        # Save the global optimal solution using Tmax
        # #[GPT-no log modify] r['x'][-1] is Tmax_nu directly.
        final_Tmax = r['x'][-1] * MUnit * AUnit
        if final_Tmax < global_Tmax:
            global_Tmax = final_Tmax
            global_lambda = r['x'].copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), optimal Tmax: {global_Tmax:.6f} N")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['MinT_original'] = records


counter: 1/1135 nfev: 216

  Final Mass: 2265.8070698773536   Tmax: 0.16709075104979923


counter: 2/1739 nfev: 234

  Final Mass: 2265.807069877136   Tmax: 0.16709075104981988


counter: 3/2261 nfev: 283

  Final Mass: 2265.807069877355   Tmax: 0.16709075104979915


counter: 4/2900 nfev: 224

  Final Mass: 2265.8070698773527   Tmax: 0.16709075104979934


counter: 5/4152 nfev: 214

  Final Mass: 2265.8070698773245   Tmax: 0.16709075104980212


Succeed to solve the problem!
Solved: 5 in 5000 (0.0010), optimal Tmax: 0.167091 N
Average number of function calls for Minpack: 38
Average number of function calls for convergent solution: 234
global_lambda: [-0.0037893095609549 -0.0004736642422803  0.0026645235102746
 -0.0024259882878317 -0.0033781461481032 -0.0000646364826165
  0.0070441988934746  0.0070441988934746]


In [10]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['MinT_original']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [11]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]


    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )


2.14 s ± 4.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]


    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )


The slowest run took 7.17 times longer than the fastest. This could mean that an intermediate result is being cached.
827 ns ± 611 ns per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]


    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )


2.16 s ± 30.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['MinT_original'] = dict(
    method='MINT, original scaling', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['MinT_original'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['MinT_original']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## MINT, rescaled


In [7]:
import twobody_MEE_hamilton_log as model
ta = model.get_integrator(formulation='MinT', compact=False)
functions = model.get_functions('MinT')
rho_func = functions['rho_func']
rho_fuel_func = functions['rho_fuel_func']
ta.pars[:] = [mu_nu, c1, c2]


In [8]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
np.random.seed(322)

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_Tmax = np.inf

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = np.empty(8)
    x0[:6] = (np.random.random(6) - 0.5) * 2.0 * 0.01
    x0[6] = np.random.random() * 0.01  # λ_m ∈ (0, 1) 

    # #[GPT-no log modify] Positive Tmax guess; last shooting variable is Tmax itself.
    # x0[7] = Tmax_nu * (0.5 + np.random.random())          # test1: [0.5*Tmax_nu, 1.5*Tmax_nu] #[GPT-no log modify]
    x0[7] = Tmax_nu * (0.5 + np.random.random()*(2.0-0.5))  # test2: [0.5*Tmax_nu, 2.0*Tmax_nu] #[GPT-no log modify]
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [9]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]

    time_start = time.perf_counter()

    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        # Call one more time with out = True
        shooting_minT(r['x'], out=True, Time_out=None)

        # Save the global optimal solution using Tmax
        # #[GPT-no log modify] r['x'][-1] is Tmax_nu directly.
        final_Tmax = r['x'][-1] * MUnit * AUnit
        if final_Tmax < global_Tmax:
            global_Tmax = final_Tmax
            global_lambda = r['x'].copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), optimal Tmax: {global_Tmax:.6f} N")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['MinT_rescaled'] = records


counter: 1/7 nfev: 225

  Final Mass: 2265.8070698773345   Tmax: 0.1670907510498009
counter: 2/8 nfev: 134

  Final Mass: 2265.8070698766937   Tmax: 0.1670907510498629


counter: 3/19 nfev: 153

  Final Mass: 2265.8070698773463   Tmax: 0.16709075104980012


counter: 4/30 nfev: 184

... (1,189 outputs left out)

  Final Mass: 2265.8070698780675   Tmax: 0.16709075104973053


counter: 734/4971 nfev: 153

  Final Mass: 2265.8070698763777   Tmax: 0.16709075104989352


Succeed to solve the problem!
Solved: 734 in 5000 (0.1468), optimal Tmax: 0.167091 N
Average number of function calls for Minpack: 72
Average number of function calls for convergent solution: 170
global_lambda: [-0.0037893095609932 -0.0004736642423605  0.0026645235102775
 -0.0024259882877585 -0.0033781461480659 -0.0000646364826164
  0.0070441988934346  0.0070441988934538]


In [10]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['MinT_rescaled']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [11]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]


    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )


5 s ± 10.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]


    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )


1.77 s ± 20.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[7] = Tmax_nu                                       # test3: Tmax_nu #[GPT-no log modify]


    r = root(
        shooting_minT,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],
            'eps': minpack_eps,
        }
    )


3.26 s ± 8.99 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['MinT_rescaled'] = dict(
    method='MINT, rescaled', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['MinT_rescaled'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['MinT_rescaled']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## MIM


In [7]:
import twobody_MEE_hamilton_log as model
ta = model.get_integrator(formulation='MIM', compact=False)
functions = model.get_functions('MIM')
rho_func = functions['rho_func']
rho_fuel_func = functions['rho_fuel_func']
ta.pars[:] = [mu_nu, c1, c2]


In [8]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
np.random.seed(322)

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_m0 = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = (np.random.random(7) - 0.5) * 2.0
    #[GPT-no log modify]
    # x0[6] = m0_nu * (0.5 + np.random.random())            # test1: [0.5*m0_nu, 1.5*m0_nu] 
    x0[6] = m0_nu * (0.5 + np.random.random()*(2.0-0.5))    # test2: [0.5*m0_nu, 2.0*m0_nu]
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [9]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[6] = m0_nu                                         # test3: m0_nu

    time_start = time.perf_counter()

    r = root(
        shooting_maxm0,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        # Call one more time with out = True
        shooting_maxm0(r['x'], out=True, Time_out=None)

        # Save the global optimal solution using Tmax
        final_m0 = r['x'][-1] * MUnit  # #[GPT-no log modify] direct m0_nu
        if final_m0 > global_m0:
            global_m0 = final_m0
            global_lambda = r['x'].copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), optimal m0: {global_m0:.6f} kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['MIM'] = records


counter: 1/1 nfev: 162

  Final Mass: 4339.3081771795405   Initial Mass: 7660.507789681453
counter: 2/4 nfev: 114

  Final Mass: 4339.30817717754   Initial Mass: 7660.507789679449


counter: 3/11 nfev: 97

  Final Mass: 4339.308177175236   Initial Mass: 7660.507789677145


counter: 4/14 nfev: 146

... (3,157 outputs left out)

counter: 1938/4995 nfev: 132

  Final Mass: 4339.308177175837   Initial Mass: 7660.507789677753
counter: 1939/4998 nfev: 146

  Final Mass: 4339.308177175219   Initial Mass: 7660.507789677128
counter: 1940/4999 nfev: 99

  Final Mass: 4339.308177175427   Initial Mass: 7660.50778967734
Succeed to solve the problem!
Solved: 1940 in 5000 (0.3880), optimal m0: 7660.507790 kg
Average number of function calls for Minpack: 102
Average number of function calls for convergent solution: 155
global_lambda: [-1.030210668658158  -0.128776482347128   0.7244117966866594
 -0.6595605283629007 -0.9184264695486967 -0.0175729094992583
  1.9151269474255646]


In [10]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['MIM']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [11]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[6] = m0_nu                                         # test3: m0_nu


    r = root(
        shooting_maxm0,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


8.67 s ± 45.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[6] = m0_nu                                         # test3: m0_nu


    r = root(
        shooting_maxm0,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


4.94 s ± 46.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()
    # Random initial guess
    x0 = initial_guess_pool[i].copy()
    # x0[6] = m0_nu                                         # test3: m0_nu


    r = root(
        shooting_maxm0,
        x0,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


3.75 s ± 18.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['MIM'] = dict(
    method='MIM', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['MIM'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['MIM']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Results


In [4]:
headings = ['Method', 'Expected first solution [ms]', 'Successful mean [ms]',
            'Failed mean [ms]', 'All-trial mean [ms]', 'Converged', 'NFEV, successful']
table = ['| ' + ' | '.join(headings) + ' |', '| ' + ' | '.join(['---'] * len(headings)) + ' |']
for row in results_by_method.values():
    values = [row['method']]
    values += [f"{row[key]:.3f}" if np.isfinite(row[key]) else '—'
               for key in ('first_ms', 'successful_ms', 'failed_ms', 'all_ms')]
    values += [f"{row['accepted']}/{row['trials']}", f"{row['nfev_successful']:.1f}"]
    table.append('| ' + ' | '.join(values) + ' |')
display(Markdown('\n'.join(table)))


| Method | Expected first solution [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Converged | NFEV, successful |
| --- | --- | --- | --- | --- | --- | --- |
| MINT, original scaling | 4276.391 | — | 4.315 | 4.276 | 5/5000 | 234.2 |
| MINT, rescaled | 68.143 | 25.274 | 7.581 | 10.003 | 734/5000 | 170.3 |
| MIM | 44.695 | 24.704 | 12.491 | 17.342 | 1940/5000 | 154.6 |